# 视频分类 CNN+LSTM

In [20]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")

In [21]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models

from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader
from pytorchvideo.models.hub import x3d_xs

from concurrent.futures import ThreadPoolExecutor, as_completed

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 一、处理数据

In [22]:
ucf_root_dir = r"/mnt/Data/daihongliang"
ucf_dataset_name = "shop2-clean"
ucf_processed_name = "shop2-Processed"
ucf_train_test_file_dir = "shopTrainTestlist"

In [23]:
def get_train_test_lists(ucf_root_dir, ucf_dataset_name, preprocess=False, tag=None, target=(256, 256), confirm=False, num_workers=8):
    data_dir = os.path.join(ucf_root_dir, ucf_dataset_name)
    class_list = os.listdir(data_dir)
    data_list = []
    preprocess_data_list = []

    for i in range(len(class_list)):
        class_item_dir = os.path.join(data_dir, class_list[i])
        data_list.extend([os.path.join(class_item_dir, it) for it in os.listdir(class_item_dir)])

    # === 多线程预处理部分 ===
    if preprocess:
        def process_video(video_path):
            try:
                # 构建输出路径
                array_save_path = video_path.replace(ucf_dataset_name, f'{ucf_dataset_name}-Processed-256')
                array_save_path = array_save_path.split('.')[0] + '.npy'
                if tag is not None:
                    array_save_path = array_save_path.split('.')[0] + f'-{tag}' + '.npy'
                
                # 如果文件已存在，直接返回路径
                if os.path.exists(array_save_path):
                    return array_save_path

                # 处理视频
                cap = cv2.VideoCapture(video_path)
                if not cap.isOpened():
                    return None

                os.makedirs(os.path.dirname(array_save_path), exist_ok=True)
                frames = []

                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame_resized = cv2.resize(frame_rgb, target)
                    frame_chw = np.transpose(frame_resized, (2, 0, 1))
                    frames.append(frame_chw)

                cap.release()
                
                # 保存处理结果
                if len(frames) > 0:
                    np.save(array_save_path, np.stack(frames))
                    return array_save_path
                else:
                    return None
                    
            except Exception as e:
                return None

        # 使用线程池并行处理
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(process_video, v) for v in data_list]
            results = []
            for future in tqdm(as_completed(futures), total=len(futures), 
                              desc="预处理进度"):
                result = future.result()
                if result:
                    results.append(result)
            
            preprocess_data_list = results

    # === 划分训练/测试集 ===
    if preprocess:
        print(f"成功处理文件数量: {len(preprocess_data_list)}/{len(data_list)}")
        if len(preprocess_data_list) == 0:
            raise ValueError("没有成功处理任何文件！")
            
        train_list, test_list = train_test_split(preprocess_data_list, test_size=0.2)
    else:
        train_list, test_list = train_test_split(data_list, test_size=0.2)
        
    return train_list, test_list

In [24]:
train_data, test_data = get_train_test_lists(
    ucf_root_dir,
    ucf_dataset_name,
    preprocess=True,
    tag="256",
    target=(256, 256),
    confirm=True
)
print("Train: ", train_data[:2], "\nTest: ", test_data[:2])

预处理进度: 100%|██████████| 3191/3191 [14:28<00:00,  3.67it/s] 

成功处理文件数量: 3191/3191
Train:  ['/mnt/Data/daihongliang/shop2-clean-Processed-256/drinking/drinking_03_08-1-256.npy', '/mnt/Data/daihongliang/shop2-clean-Processed-256/hiding/video15-256.npy'] 
Test:  ['/mnt/Data/daihongliang/shop2-clean-Processed-256/tearing/video-22-256.npy', '/mnt/Data/daihongliang/shop2-clean-Processed-256/drinking/10月12日3142523525-1-256.npy']


In [25]:
classes = {}
classes_file = os.path.join(ucf_root_dir, ucf_train_test_file_dir, 'classshop.txt')
with open(classes_file) as fin:
    for row in fin:
        line_info = row.strip().split(' ')
        classes[line_info[1]] = int(line_info[0]) - 1
list(classes.items())[:2]

[('drinking', 0), ('eating', 1)]

## 二、编写 Torch 的 Dataset

In [26]:
class UCFDataset(Dataset):
    def __init__(self, data, classes):
        self.num_frames = 64

        self.data = data
        self.classes = classes

        # 数据处理
        self.transform = v2.Compose(
            [
                v2.Resize(256),
                v2.CenterCrop(224),
                v2.ConvertImageDtype(torch.float32),
                v2.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
           ]
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        np_data_path = self.data[i]
        np_data = np.load(np_data_path)

        total_frames = np_data.shape[0]

        # 选择帧
        if total_frames <= self.num_frames:
            sample_data = np_data
        else:
            max_start = total_frames - self.num_frames
            start_frame = random.randint(0, max_start)
            sample_data = np_data[start_frame: start_frame + self.num_frames, :, :, :]

        frames = torch.tensor(sample_data).float()
        frames = self.transform(frames)

        label = self.classes[np_data_path.split('/')[-2]]
        return frames, label

In [27]:
trainset = UCFDataset(train_data, classes)
testset = UCFDataset(test_data, classes)
d, l = trainset.__getitem__(123)
print(d.shape, l)

torch.Size([43, 3, 224, 224]) 3


## 三、编写 Torch 的 DataLoader

In [28]:
def collate_fn(batch):
    data = []
    label = []

    # 筛选总帧数小于64的样本
    for d, l in batch:
        num_frames = d.shape[0]
        if num_frames < 64:
            continue  # 跳过帧数不足的样本

        indices = torch.linspace(0, num_frames - 1, 16).long()
        sampled_frames = d[indices]

        data.append(sampled_frames)
        label.append(l)

    # 处理批量中无有效样本的情况
    if len(data) == 0:
        return torch.tensor([]), torch.tensor([])

    # 组装张量（所有样本采样后帧数相同）
    data_tensor = torch.stack(data, dim=0).type(torch.FloatTensor).contiguous()
    label_tensor = torch.tensor(label).type(torch.FloatTensor)

    return data_tensor, label_tensor

In [29]:
trainloader = DataLoader(trainset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=8)
testloader = DataLoader(testset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=8)

batch_data, batch_labels = next(iter(trainloader))
print("数据形状:", batch_data.shape)
print("标签形状:", batch_labels.shape)

数据形状: torch.Size([6, 16, 3, 224, 224])
标签形状: torch.Size([6])


## 四、设计网络

In [30]:
class CNNLSTM(nn.Module):
    def __init__(self, num_classes, num_frames=10, lstm_hidden_size=256, lstm_num_layers=2):
        super(CNNLSTM, self).__init__()
        self.num_frames = num_frames
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_num_layers = lstm_num_layers
        
        # 使用预训练的ResNet18作为CNN特征提取器[7](@ref)
        cnn_model = models.resnet18(pretrained=True)
        
        # 移除最后的全连接层，保留特征提取部分
        self.cnn = nn.Sequential(*list(cnn_model.children())[:-1])
        
        # 获取CNN输出特征维度
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 224, 224)
            cnn_output_dim = self.cnn(dummy_input).view(1, -1).size(1)
        
        # LSTM层用于时序建模[4](@ref)
        self.lstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            batch_first=True,
            dropout=0.3 if lstm_num_layers > 1 else 0
        )
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(lstm_hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        # x形状: [batch_size, num_frames, channels, height, width]
        batch_size, num_frames, C, H, W = x.size()
        
        # 重塑为: [batch_size * num_frames, channels, height, width]
        cnn_input = x.view(batch_size * num_frames, C, H, W)
        
        # CNN特征提取
        cnn_features = self.cnn(cnn_input)
        cnn_features = cnn_features.view(batch_size * num_frames, -1)
        
        # 重塑为LSTM输入格式: [batch_size, num_frames, feature_dim]
        lstm_input = cnn_features.view(batch_size, num_frames, -1)
        
        # LSTM处理时序信息[3](@ref)
        lstm_out, (hidden, cell) = self.lstm(lstm_input)
        
        # 取最后一个时间步的输出
        last_output = lstm_out[:, -1, :]
        
        # 分类
        output = self.classifier(last_output)
        
        return output

In [31]:
net = CNNLSTM(4)
o = net(batch_data)
o.shape

torch.Size([6, 4])

## 五、构建训练代码

In [32]:
seed = 1
torch.manual_seed(seed)  # 为CPU设置随机种子
torch.cuda.manual_seed(seed)  # 为当前GPU设置随机种子
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU，为所有GPU设置随机种子
np.random.seed(seed)  # Numpy module.
random.seed(seed)  # Python random module.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [33]:
save_root = "./checkpoints_shop/CNNLSTM"
num_classes = 4
learning_rate = 1e-4
train_epochs = 100
lr_scheduler_step = 50

In [34]:
# 创建检查目录
os.makedirs(os.path.join(save_root, 'acc'), exist_ok=True)
os.makedirs(os.path.join(save_root, 'loss'), exist_ok=True)

In [35]:
model = CNNLSTM(num_classes)
if torch.cuda.device_count() > 1:
    print(f"使用 {torch.cuda.device_count()} 个GPU进行训练。")
    model = nn.DataParallel(model)  # 包装模型
model = model.cuda()

使用 4 个GPU进行训练。


In [36]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=lr_scheduler_step, gamma=0.1)
criterion = nn.CrossEntropyLoss()

In [37]:
def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    return accuracy, precision, recall, micro_f1, macro_f1

In [38]:
best_accuracy = 0
min_loss = float('inf')

for epoch in range(1, train_epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{train_epochs}")
    print(f"{'='*60}")
    
    # 更新学习率
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"当前学习率: {current_lr}")

    # 训练循环
    model.train()
    train_loss, train_correct, train_num = 0.0, 0, 0
    train_pred, train_label = [] , []
    for _, (x, y) in enumerate(tqdm(trainloader, desc="训练中")):
        if not x.shape[0]:
            continue
        x, y = x.cuda(), y.long().cuda()
        
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(x)
        loss = criterion(outputs, y)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        # 统计信息
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_correct += (predicted == y).sum().item()
        train_num += y.size(0)
        
        train_pred.extend(predicted.cpu().numpy())
        train_label.extend(y.cpu().numpy())

    train_avg_loss = train_loss / len(trainloader)
    train_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(train_label, train_pred)
    print(f"Loss: {train_avg_loss:.4f}, Accuracy: {train_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

    # 测试循环
    model.eval()
    test_loss, test_correct_num, test_num = 0.0, 0, 0
    test_pred, test_label = [], []
    
    with torch.no_grad():
        for _, (x, y) in enumerate(tqdm(testloader, desc="测试中")):
            if not x.shape[0]:
                continue
            x, y = x.cuda(), y.long().cuda()
            
            outputs = model(x)
            loss = criterion(outputs, y)
            
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_correct_num += (predicted == y).sum().item()
            test_num += y.size(0)
            
            test_pred.extend(predicted.cpu().numpy())
            test_label.extend(y.cpu().numpy())
    
    test_avg_loss = test_loss / len(testloader)
    test_accuracy, precision, recall, micro_f1, macro_f1 = calculate_metrics(test_label, test_pred)
    print(f"Loss: {test_avg_loss:.4f}, Accuracy: {test_accuracy:.2%}, Precision: {precision:.4f}, Recall: {recall:.4f}, Micro F1: {micro_f1:.4f}, Macro F1: {macro_f1:.4f}")

     # 保存最佳准确率模型
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        save_path = os.path.join(save_root, 'acc', f'best_acc.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
    
    # 保存最低损失模型
    if test_avg_loss < min_loss:
        min_loss = test_avg_loss
        save_path = os.path.join(save_root, 'loss', f'best_loss.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")

    
    # 正确率超过99%时结束训练
    if test_accuracy >= 0.99:
        print(f"训练准确率已达到 {test_accuracy:.2%}，提前终止训练")
        save_path = os.path.join(save_root, f'early_stop_epoch_{epoch}.pth')
        torch.save(model.state_dict(), save_path)
        print(f"模型已保存: {save_path} (Epoch {epoch}, Acc: {test_accuracy:.2%}, Loss: {test_avg_loss:.4f})")
        break


/home/wuhao@kemande.cn/anaconda3/envs/C3d_new/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:139: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "



Epoch 1/100
当前学习率: 0.0001


训练中:   0%|          | 0/319 [00:00<?, ?it/s]

训练中: 100%|██████████| 319/319 [01:39<00:00,  3.19it/s]


Loss: 1.3119, Accuracy: 33.69%, Precision: 0.2489, Recall: 0.3369, Micro F1: 0.3369, Macro F1: 0.2001


...

Loss: 1.4909, Accuracy: 77.34%, Precision: 0.7860, Recall: 0.7734, Micro F1: 0.7734, Macro F1: 0.7601

Epoch 100/100
当前学习率: 1.0000000000000002e-06


训练中: 100%|██████████| 319/319 [04:10<00:00,  1.27it/s]


Loss: 0.0017, Accuracy: 100.00%, Precision: 1.0000, Recall: 1.0000, Micro F1: 1.0000, Macro F1: 1.0000


测试中: 100%|██████████| 80/80 [00:43<00:00,  1.86it/s]

Loss: 1.6860, Accuracy: 75.82%, Precision: 0.7765, Recall: 0.7582, Micro F1: 0.7582, Macro F1: 0.7478
